# Advanced 08 — HyDE: Imagine a Document Before Searching

## Harborline Support scenario

Employees describe problems informally, while Harborline's documentation uses formal identity, finance, and policy terminology. We will test whether a document-shaped hypothetical search representation closes that gap.

**Success criteria**

- improve top-1 retrieval on the labelled semantic-gap slice;
- preserve exact identifiers and numerical lookups through routing;
- keep hypotheses out of the real-evidence ledger; and
- compare quality against the extra hypothesis count used as a transparent cost proxy.

**Safety boundary:** the hypothetical document is untrusted search text. It is never evidence and is never cited. Authorization is applied before every retrieval path.

## Learning objectives and prerequisites

You should already understand lexical, dense, and hybrid retrieval; query rewriting; reranking; and Recall@k/MRR. By the end, you will be able to implement and evaluate single-hypothesis, fused, original-plus-HyDE, and conditional HyDE paths.

This notebook is credential-free. Production HyDE uses an instruction-following model plus a dense document encoder. The lab substitutes a deterministic hypothesis fixture and transparent TF-IDF vectors so the representation change and its failures remain inspectable. These vectors are a teaching proxy, not a claim about production embedding quality.

## Architecture walkthrough

```text
query
  ├─ original path ─────────────────────┐
  └─ hypothesis generator → search text ├→ retrieve REAL documents
                                       ↓
                                  fuse / rerank
                                       ↓
                                real-evidence ledger
```

The lab trace exposes the original query, selected strategy, hypothetical search texts, ranked real documents, and evidence IDs. It does not expose or require private chain-of-thought.

In [ ]:
from pathlib import Path
import sys

module_dir = Path("curriculum/advanced/08-hyde-retrieval")
if not module_dir.exists():
    module_dir = Path.cwd()
sys.path.insert(0, str(module_dir.resolve()))

from lab import (
    CORPUS, EVALUATION_CASES, authorized_documents, evaluate,
    generate_hypotheses, retrieve, route_query, summarize,
)

print(f"Loaded {len(CORPUS)} documents and {len(EVALUATION_CASES)} labelled cases.")

## Inspect the corpus and query slices

Before comparing methods, inspect what the evaluator considers relevant. The set contains three semantic-gap questions, two exact identifiers, and one numerical lookup. A useful method should not improve one slice by silently damaging another.

In [ ]:
for case in EVALUATION_CASES:
    print(f"{case.query_type:18} | {case.query:55} | expected={sorted(case.relevant_ids)}")

## Baseline: search with the literal query

The first query shares superficial words with an application-logging document but uses none of the formal identity terms in the correct document. This makes the query–document gap visible.

In [ ]:
query = "Why does the app log me out overnight?"
baseline = retrieve(query, strategy="original", top_k=3)
for item in baseline.results:
    print(item.rank, item.document.doc_id, round(item.score, 3))

The baseline is transparent, but the literal words point toward logging rather than token lifecycle. Now generate corpus-shaped search text.

In [ ]:
hypotheses = generate_hypotheses(query, count=2)
for index, hypothesis in enumerate(hypotheses, start=1):
    print(f"Hypothesis {index}: {hypothesis}")

hyde = retrieve(query, strategy="hyde", top_k=3, hypothesis_count=2)
print("Retrieved real evidence:", hyde.evidence_ids)

## Verify the evidence boundary

The generated passages should influence ranking but can never become evidence. The next assertion checks every evidence ID against the authorized corpus.

In [ ]:
authorized_ids = {doc.doc_id for doc in authorized_documents(CORPUS, "harborline")}
assert set(hyde.evidence_ids) <= authorized_ids
assert not set(hyde.hypotheses) & set(hyde.evidence_ids)
print("Evidence invariant holds: every evidence ID names a real authorized document.")

## Experiment 1 — Compare retrieval policies

We now hold the corpus, vectorizer, labels, and top-1 budget constant. The only change is the search representation and routing policy. `total_hypotheses` is not monetary cost; it is an explicit count of added generation work.

In [ ]:
strategies = ("original", "hyde", "hyde_plus_original", "conditional")
reports = {}
for strategy in strategies:
    rows = evaluate(strategy, top_k=1, hypothesis_count=2)
    reports[strategy] = summarize(rows)
    print(strategy, reports[strategy])

assert reports["conditional"]["mean_recall_at_k"] >= reports["original"]["mean_recall_at_k"]

Interpret the output rather than choosing the row with the most sophisticated name. Universal HyDE can help semantic-gap questions while adding avoidable work or damaging an exact proprietary lookup. The conditional policy is designed to preserve exact and numerical paths.

## Experiment 2 — Slice analysis

A single average can hide regressions. Compare the original baseline, universal HyDE, and conditional policy case by case.

In [ ]:
for strategy in ("original", "hyde", "conditional"):
    print(f"\n{strategy.upper()}")
    for row in evaluate(strategy, top_k=1, hypothesis_count=1):
        print(
            f"{row.query_type:18} recall={row.recall_at_k:.0f} "
            f"route={row.strategy:8} top={row.retrieved_ids[0]}"
        )

## Failure injection — a confident wrong hypothesis

`ZX-47` is an internal hardware controller. The deterministic generator intentionally imagines a pharmaceutical compound, reproducing a core HyDE risk: generation injects a meaning before grounding.

In [ ]:
ambiguous = "What is ZX-47?"
wrong_hyde = retrieve(ambiguous, strategy="hyde", top_k=2)
safe_route = retrieve(ambiguous, strategy="conditional", top_k=2)

print("Hypothesis:", wrong_hyde.hypotheses[0])
print("Universal HyDE:", wrong_hyde.evidence_ids)
print("Conditional route:", route_query(ambiguous), safe_route.evidence_ids)
assert safe_route.evidence_ids[0] == "zx-47-controller"

The mitigation is architectural: exact identifiers stay on the original/exact path. A prompt cannot supply proprietary knowledge the generator does not have.

## Experiment 3 — Multi-hypothesis fusion

The lab retrieves separately for each hypothesis and applies Reciprocal Rank Fusion. This preserves each ranking instead of averaging potentially different meanings into one vector. Compare one and two hypotheses while keeping `top_k` fixed.

In [ ]:
for count in (1, 2):
    trace = retrieve(
        "Can I look after my parent while staying overseas?",
        strategy="hyde_plus_original",
        top_k=3,
        hypothesis_count=count,
    )
    print(f"hypotheses={count} results={trace.evidence_ids}")

More hypotheses are not automatically better. They increase generation and retrieval work; distinct but wrong interpretations can also increase noise. Use a labelled set to choose the count and fusion method.

## Security test — authorization before relevance

The corpus contains a highly similar Tenant B identity document. It must not enter the candidate set, even when HyDE creates matching terminology.

In [ ]:
security_trace = retrieve(query, strategy="hyde_plus_original", top_k=10, hypothesis_count=2)
assert "tenant-b-secret" not in security_trace.evidence_ids
print("Authorized candidate IDs:", security_trace.evidence_ids)

## Production upgrade path

| Lab | Production | Keep invariant |
|---|---|---|
| deterministic hypothesis fixture | approved local/hosted instruction model | bounded count, length, timeout, cost |
| TF-IDF vector proxy | evaluated document encoder | version compatibility and offline regression |
| in-memory search | filtered ANN/hybrid index | authorization before every leg |
| rules | calibrated classifier or policy/model router | explicit safe baseline and clarify route |
| local RRF | service/store fusion plus reranker | stable IDs and per-leg trace |
| fixture cases | versioned representative evaluation set | per-slice gates and changed-failure review |

Version the generator prompt/model, encoder, corpus snapshot, index, router, fusion, reranker, and evaluator together. Roll out to a small routed slice and retain a kill switch to the original baseline.

## Exercises

1. Add a semantic-gap query and relevant document, then predict each strategy before running it.
2. Add a missing-document case. Verify that no generated hypothesis can become a valid evidence ID.
3. Implement mean-vector aggregation and compare it with RRF on two opposing hypotheses.
4. Add a `clarify` route for ambiguous short entity queries without exact identifiers.
5. Replace the fixture generator with a local model adapter while preserving the bounded return type.
6. Define a release gate covering semantic-gap Recall@10, exact-identifier regressions, unauthorized candidates, p95 latency, and cost.

## Summary

HyDE is a query-side retrieval transformation inside RAG. It can bridge user language and document language, but it adds a generator call and a pre-grounding interpretation risk. Keep the hypothetical text outside the evidence chain, compare against the original-query baseline, evaluate by query slice, and route HyDE only where measured gains justify its cost and failure modes.